# 🍵 Corndel AI6 "TeaTime Tycoon": A Practical Stats Adventure
### For AI/ML Engineering Apprentices · (UK Level 6) · ⏱ ~2 hours

---

You've just joined the data team at **Steepify**, the scrappy startup behind **TeaTime Tycoon** — a tea-shop management game that somehow has 50,000 downloads and a devoted following of people who take their Earl Grey *very* seriously.

It's 9:03am on a Monday. Your product manager, Priya, Slacks you:

> *"Morning! Quick one — I've been eyeballing our ratings over the weekend and I'm convinced App Store users are way harsher than Play Store users. Can you just confirm that's true so we can escalate to the iOS team?"*

Before Priya books a sprint and a designer and causes a panic, you need to answer a precise question with data:

> **Is the difference in ratings between the two platforms real, or is it just noise?**

---

### What you'll build in this notebook

By the end, you'll be able to:

- Frame a data question as a proper **hypothesis test**
- Load and validate data like a professional
- Run and interpret a **Welch's t-test**
- Understand what a **p-value actually means** (and avoid the most common misconceptions)
- Measure **effect size** to distinguish statistical from practical significance
- Build a **permutation test from scratch** — which will give you the deepest intuition for p-values you'll ever get
- Understand **statistical power** through a hands-on subsampling experiment
- Produce a **business-ready summary** that translates statistics into action

---

> 💡 **How to use this notebook:** Read every markdown cell — they do the teaching. The code cells show you how to implement it. Run each cell in order, and don't skip the 🔧 **Try It** prompts at the end of each section.


---
## ⚙️ Setup

Nothing exotic. We'll use the standard scientific Python stack throughout.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy
from scipy import stats
from IPython.display import display

# ── Plot styling (pure matplotlib — no seaborn dependency) ────────────────
plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.35,
    "axes.facecolor":    "#fafafa",
    "figure.facecolor":  "white",
    "axes.prop_cycle":   plt.cycler(color=["#e07b54","#5b8db8","#2ecc71","#9b59b6"]),
})

COLOURS = {"App Store": "#e07b54", "Play Store": "#5b8db8"}

# ── Simulation config — turn up for publication-quality results ───────────
N_PERMUTATIONS = 3_000   # increase to 10_000 for more precise p-values (~10s extra)
N_POWER_TRIALS = 300     # increase to 500 for a smoother power curve

print(f"Python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}  |  numpy  {np.__version__}  |  scipy  {scipy.__version__}")
print()
print(f"Simulation settings:")
print(f"  N_PERMUTATIONS = {N_PERMUTATIONS}  (edit above to increase precision)")
print(f"  N_POWER_TRIALS = {N_POWER_TRIALS}  (edit above for smoother power curve)")
print()
print("✅ Ready. Let's investigate Priya's hunch.")


---
## Step 1 — Data Engineering: Building the Pipeline

In a real job, you'd scrape these reviews using something like `google-play-scraper` or the App Store Connect API. For this notebook we'll synthesise a realistic dataset — but we'll do it properly.

**The pattern we're using:**
- If real CSV files already exist in `data/` → we use them (swap in your own scraped data!)
- Otherwise → we generate synthetic data, save it, then read it back

This is good engineering habit: your analysis notebook should be *data-agnostic*. The pipeline stays the same whether the data is synthetic or real.

**What we're building into the data:**
- Discrete 1–5 star ratings (like real app store data — not continuous!)
- Slightly different distributions per platform (baked-in, but subtle)
- Review text, IDs, and dates — because real data always has extra columns
- Unequal sample sizes (250 App Store, 260 Play Store) — again, realistic


In [ ]:
# ── Synthetic review text generator ──────────────────────────────────────
def make_review_text(rating: int, rng: np.random.Generator) -> str:
    """Returns a plausible fake review for a given star rating."""
    bank = {
        5: ["Absolutely obsessed. My commute is now just tea strategy.",
            "Smooth, fast, and genuinely addictive. 10/10 would steep again.",
            "Best tycoon game on the market. The kettle upgrade arc is *chef's kiss*.",
            "My therapist says I need to stop but the leaderboard disagrees."],
        4: ["Really fun, couple of small bugs but nothing game-breaking.",
            "Solid app. Would love a chamomile expansion pack.",
            "Great concept, UI could be cleaner in places.",
            "Works well most of the time. The tutorial is a bit long."],
        3: ["It's okay. Does the job but nothing special.",
            "Average experience. The tea puns get old fast.",
            "Mixed bag — some bits great, some bits clunky.",
            "Fine. I've seen better tycoon games. I've seen worse."],
        2: ["Keeps lagging during the lunchtime rush event.",
            "Not as good as I hoped. Too many microtransactions for sugar.",
            "Frustrating UI decisions. Why can't I sort by tea type?",
            "Needs a lot of polish — feels unfinished."],
        1: ["Crashes every time I try to unlock Oolong. Unplayable.",
            "Complete disaster. Lost all my progress twice.",
            "Unusable on my device. Deleted after 10 minutes.",
            "One star is too generous."]
    }
    return rng.choice(bank[int(rating)])


# ── Dataset generator ─────────────────────────────────────────────────────
def generate_teatime_data(
    app_path: Path,
    play_path: Path,
    n_app: int = 250,
    n_play: int = 260,
    seed: int = 42,
) -> None:
    """
    Synthesises two realistic review datasets and saves them as separate CSVs.

    App Store: slightly more critical (we bake this in deliberately so the 
    notebook has something real to find — but the difference is subtle enough
    that you wouldn't be certain without the test).

    Play Store: slightly more positive.
    """
    rng = np.random.default_rng(seed)

    app_probs  = [0.05, 0.07, 0.18, 0.35, 0.35]   # 1→5 stars
    play_probs = [0.04, 0.05, 0.13, 0.35, 0.43]

    app_ratings  = rng.choice([1, 2, 3, 4, 5], size=n_app,  p=app_probs)
    play_ratings = rng.choice([1, 2, 3, 4, 5], size=n_play, p=play_probs)

    # Dates: random days within a 90-day window
    start = np.datetime64("2025-11-26")
    app_dates  = (rng.integers(0, 90, size=n_app ).astype("timedelta64[D]") + start).astype(str)
    play_dates = (rng.integers(0, 90, size=n_play).astype("timedelta64[D]") + start).astype(str)

    pd.DataFrame({
        "review_id":   [f"AS-{i:04d}" for i in range(1, n_app  + 1)],
        "platform":    "App Store",
        "rating":      app_ratings.astype(int),
        "date":        app_dates,
        "review_text": [make_review_text(r, rng) for r in app_ratings],
    }).to_csv(app_path, index=False)

    pd.DataFrame({
        "review_id":   [f"PS-{i:04d}" for i in range(1, n_play + 1)],
        "platform":    "Play Store",
        "rating":      play_ratings.astype(int),
        "date":        play_dates,
        "review_text": [make_review_text(r, rng) for r in play_ratings],
    }).to_csv(play_path, index=False)


# ── Load or generate ──────────────────────────────────────────────────────
data_dir  = Path("data")
data_dir.mkdir(exist_ok=True)
app_path  = data_dir / "app_store_reviews.csv"
play_path = data_dir / "play_store_reviews.csv"

if app_path.exists() and play_path.exists():
    print("📂 Found existing CSV files — using those.")
    print("   (To regenerate synthetic data, delete the files in data/ and rerun.)")
else:
    print("📂 No CSV files found → generating synthetic dataset...")
    generate_teatime_data(app_path, play_path)
    print(f"   Saved: {app_path}")
    print(f"   Saved: {play_path}")

app_df  = pd.read_csv(app_path)
play_df = pd.read_csv(play_path)

print(f"\n✅ Loaded {len(app_df)} App Store reviews and {len(play_df)} Play Store reviews.")
print("\nHere's a sample from each:")
display(app_df.sample(3, random_state=1))
display(play_df.sample(3, random_state=1))


---
## Step 2 — Sanity Checks: Never Trust Data You Haven't Inspected

This step gets skipped more often than it should. Before running any analysis, verify:

- Are ratings actually in the range 1–5?
- Are there any missing values?
- Are platform labels what we expect?
- Do the sample sizes look right?

These checks take two minutes and have saved many engineers from presenting embarrassing nonsense. Run them every time.


In [ ]:
def sanity_check(df: pd.DataFrame, name: str) -> None:
    print(f"{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Rows           : {len(df)}")
    print(f"  Missing values : {df.isnull().sum().sum()}")
    print(f"  Rating range   : {df['rating'].min()} – {df['rating'].max()}")
    print(f"  Unique platforms: {df['platform'].unique()}")
    print(f"  Date range     : {df['date'].min()} → {df['date'].max()}")
    print()

sanity_check(app_df,  "App Store Reviews")
sanity_check(play_df, "Play Store Reviews")

# Check for any ratings outside 1-5 (would be a data quality issue)
bad_app  = app_df[~app_df['rating'].between(1, 5)]
bad_play = play_df[~play_df['rating'].between(1, 5)]

if len(bad_app) + len(bad_play) == 0:
    print("✅ All ratings are in the valid 1–5 range. No data quality issues found.")
else:
    print(f"⚠️  Found {len(bad_app) + len(bad_play)} out-of-range ratings — investigate before continuing!")


---
## Step 3 — Exploratory Data Analysis

> *"Never run a statistical test blind."*

Visualise first. Calculate second. This order matters because:
1. You'll catch data problems that summary stats miss
2. You'll build intuition for whether any difference *looks* real before the maths starts
3. You might spot something unexpected that changes your question entirely

Look at the charts below and ask yourself: does the difference between the platforms *feel* like signal or noise?


In [ ]:
# ── Combine into one tidy DataFrame ──────────────────────────────────────
df = pd.concat([app_df, play_df], ignore_index=True)

# ── Summary statistics ─────────────────────────────────────────────────────
summary = (
    df.groupby("platform")["rating"]
      .agg(n="count", mean="mean", median="median", std="std")
      .round(3)
)
print("📊 Summary Statistics")
print("─" * 40)
display(summary)

raw_diff = summary.loc["App Store", "mean"] - summary.loc["Play Store", "mean"]
print(f"\nRaw difference in means (App Store − Play Store): {raw_diff:+.3f} stars")
print("↑ This is what Priya spotted. But is it meaningful? Let's find out.")


In [ ]:
# ── Figure 1: Rating distributions side by side ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, platform in zip(axes, ["App Store", "Play Store"]):
    subset = df[df["platform"] == platform]["rating"]
    counts = subset.value_counts().sort_index()
    colour = COLOURS[platform]
    
    bars = ax.bar(counts.index, counts.values,
                  color=colour, edgecolor="white", linewidth=1.8, alpha=0.9, zorder=3)
    
    mean_val = subset.mean()
    ax.axvline(mean_val, color="#2c3e50", linestyle="--", linewidth=2,
               label=f"Mean = {mean_val:.2f}", zorder=4)
    
    # Annotate each bar with its count
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                str(count), ha="center", va="bottom", fontsize=10, color="#444")
    
    ax.set_title(platform, fontsize=15, fontweight="bold", color=colour, pad=10)
    ax.set_xlabel("Star Rating", fontsize=12)
    ax.set_ylabel("Number of Reviews", fontsize=12)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_ylim(0, max(counts.values) * 1.2)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.4)

plt.suptitle("TeaTime Tycoon — How Are We Being Rated?", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print("👆 The means look different, but the distributions overlap substantially.")
print("   This is exactly the situation where gut-feel fails and statistics helps.")


In [ ]:
# ── Figure 2: Side-by-side box + strip plot ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

# Box plots
bp = ax.boxplot(
    [df[df["platform"] == p]["rating"].values for p in ["App Store", "Play Store"]],
    positions=[1, 2], widths=0.45, patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.5), capprops=dict(linewidth=1.5),
    flierprops=dict(marker="o", markersize=4, alpha=0.3)
)

for patch, platform in zip(bp["boxes"], ["App Store", "Play Store"]):
    patch.set_facecolor(COLOURS[platform])
    patch.set_alpha(0.7)

# Strip plots (jittered individual points)
for i, platform in enumerate(["App Store", "Play Store"], start=1):
    y = df[df["platform"] == platform]["rating"].values
    x = np.random.default_rng(seed=99).normal(i, 0.07, size=len(y))
    ax.scatter(x, y, alpha=0.2, s=18, color=COLOURS[platform], zorder=2)

ax.set_xticks([1, 2])
ax.set_xticklabels(["App Store", "Play Store"], fontsize=13)
ax.set_ylabel("Star Rating", fontsize=12)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title("Rating Distribution — Every Review Visible", fontsize=13)

# Custom legend
patches = [mpatches.Patch(color=v, label=k, alpha=0.7) for k, v in COLOURS.items()]
ax.legend(handles=patches, fontsize=11)
plt.tight_layout()
plt.show()

print("💡 Each dot is one review. The box shows median, IQR, and whiskers (1.5×IQR).")
print("   Notice how much the two groups overlap — this is why we need a test.")


---
## Step 4 — Setting the Hypotheses (Before We Touch the Test)

This step is non-negotiable. Writing hypotheses **before** running anything:

1. Forces you to be precise about what question you're actually answering
2. Prevents you from unconsciously searching for significance after seeing the data
3. Determines which test is appropriate

---

### The Null and Alternative Hypotheses

| | What it claims |
|---|---|
| **H₀ (null)** | There is no real difference in mean ratings. Any observed gap is random sampling noise. μ_AS = μ_PS |
| **H₁ (alternative)** | There is a real difference. The means are genuinely different. μ_AS ≠ μ_PS |

We're using **≠** (two-tailed) rather than < or >. Why? Because we're being honest — we don't actually know which direction the difference goes. If Priya had given us a strong prior reason to test only one direction, we'd use a one-tailed test. Two-tailed is the safer, more conservative default.

---

### The Significance Level (α)

Before seeing any result, we set our threshold. We'll use **α = 0.05** — the conventional default across product analytics and A/B testing.

This means: we're willing to accept a **5% chance of a false positive** — i.e. wrongly claiming a real difference when there isn't one (a Type I error).

> ⚠️ α is not a magic number. In safety-critical contexts (medical devices, financial risk) you'd use 0.01 or 0.001. In exploratory product analytics you might relax to 0.10. Choosing α is a **business decision**, not a statistical one.


In [ ]:
ALPHA = 0.05

print("Hypothesis Test Setup")
print("─" * 40)
print("H₀ : μ_AS = μ_PS   (no real difference)")
print("H₁ : μ_AS ≠ μ_PS   (a real difference exists)")
print(f"α  : {ALPHA}  (two-tailed test)")
print("\nDecision rule:")
print(f"  If p-value < {ALPHA}  → reject H₀  (significant result)")
print(f"  If p-value ≥ {ALPHA}  → fail to reject H₀  (insufficient evidence)")


---
## Step 5 — Welch's T-Test

We're using an **independent samples t-test** — "independent" because the two groups are entirely different people. (A *paired* t-test is for when the same people appear in both groups, like a before/after study.)

### Why Welch's, not Student's?

The classic Student's t-test assumes both groups have the **same variance**. That's often unrealistic — different platforms attract different demographics, different devices, different feedback cultures.

**Welch's t-test** relaxes this assumption. It adjusts the degrees of freedom based on the observed variances. There's essentially no downside to using it over Student's, so it's the right default.

In : this is simply .

---

### "But wait — star ratings are 1 to 5. Is a t-test even valid?"

Good question. Star ratings are **discrete** (only whole numbers) and **ordinal** (the gap between 3 and 4 stars isn't necessarily the same as between 4 and 5). The t-test was designed for continuous, normally distributed data. So what gives?

The answer is the **Central Limit Theorem (CLT)**:

> *For sufficiently large samples, the distribution of the **sample mean** approaches a normal distribution — regardless of the shape of the underlying data.*

With n = 250+ reviews per platform, we're not testing individual ratings against a normal distribution. We're comparing **means**, and those means are normally distributed by the CLT. The t-test is applied to those means, not to the raw ratings.

In practice, teams t-test star rating means constantly and it works well at this sample size. That said, if you're ever working with very small samples (n < 30) or heavily skewed ratings, you should prefer a non-parametric test like Mann-Whitney U instead — we cover that in the exercises.

---

### What the test weighs up

Three things determine how convincing a difference is:

1. **The gap between the means** — a bigger gap is harder to explain as chance
2. **The spread (standard deviation)** of each group — noisy data makes it harder to detect real differences
3. **The sample size** — more data → more confidence → easier to detect smaller differences

The t-statistic combines all three. The p-value then tells us how surprising that t-statistic would be if H₀ were true.


In [ ]:
app_ratings  = app_df["rating"].values
play_ratings = play_df["rating"].values

# ── Welch's t-test ────────────────────────────────────────────────────────
t_stat, p_value = stats.ttest_ind(app_ratings, play_ratings, equal_var=False)

# Variances (useful to see)
print("Observed Variances")
print(f"  App Store  : {np.var(app_ratings,  ddof=1):.4f}")
print(f"  Play Store : {np.var(play_ratings, ddof=1):.4f}")
print("  (These are similar but not identical — Welch's handles this correctly)")
print()
print("Welch's T-Test Result")
print("─" * 35)
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {p_value:.4f}")


---
## Step 6 — What Does the P-Value Actually Mean?

The p-value is the single most misunderstood number in data science. Let's fix that.

> **The p-value is the probability of observing a difference at least this large, *assuming H₀ is true*.**

In plain English: *"If there was genuinely no difference between the platforms, how often would random sampling give us a gap this big by accident?"*

A small p-value means: this result would be very surprising in a world where H₀ is true. So either H₀ is true and we got unlucky, or H₀ is false. At p < 0.05 we conventionally bet on the latter.

---

### The Myth-Busting Table

| ❌ Common (wrong) interpretation | ✅ What it actually means |
|---|---|
| "p = 0.03 means there's a 3% chance H₀ is true" | p is calculated *assuming* H₀ is true. It says nothing about the probability of H₀ |
| "p > 0.05 proves there's no difference" | It means we lack enough evidence to reject H₀. Absence of evidence ≠ evidence of absence |
| "p < 0.05 means the effect is important" | Significance just means the difference is probably real. Effect size tells you if it matters |
| "p = 0.049 is significant but p = 0.051 isn't" | This is an artefact of an arbitrary threshold. Both are weak evidence either way |


In [ ]:
print("─" * 45)
print("  Significance Test Result")
print("─" * 45)
print(f"  p-value = {p_value:.4f}")
print(f"  α       = {ALPHA}")
print()

if p_value < ALPHA:
    print(f"  ✅  p ({p_value:.4f}) < α ({ALPHA})")
    print("  → We REJECT H₀")
    print("  → The difference in ratings is statistically significant.")
    print("  → Something real is probably driving this gap.")
else:
    print(f"  🚫  p ({p_value:.4f}) ≥ α ({ALPHA})")
    print("  → We FAIL TO REJECT H₀")
    print("  → The observed gap could plausibly be random noise.")
    print("  → Tell Priya not to panic — yet.")

print()
print("But wait — significant doesn't mean *important*. Keep reading.")


---
## Step 7 — Effect Size: *Should We Actually Care?*

Here's a trap that catches even experienced analysts:

> **With enough data, almost anything becomes statistically significant.**

With 50,000 reviews, a difference of 0.02 stars will have p < 0.001. It's "real" — but is it worth engineering time?

This is why you always report **effect size** alongside the p-value. Effect size measures the *magnitude* of the difference in a standardised, scale-independent way.

---

### Cohen's d

The most widely used effect size for comparing two means:

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}$$

It answers: *how many pooled standard deviations apart are these two groups?*

### Hedges' g

Cohen's d has a known upward bias when sample sizes are small. **Hedges' g** applies a small correction factor J to reduce this bias. With n > 100 per group the difference is tiny, but it costs nothing to report it — and it signals that you know what you're doing.

---

### Rough interpretation (Cohen, 1988)

| d (or g) | Typical reading |
|---|---|
| < 0.2 | Negligible — probably not worth acting on alone |
| 0.2 – 0.5 | Small — real but modest |
| 0.5 – 0.8 | Medium — worth investigating |
| > 0.8 | Large — this is a meaningful gap |

> ⚠️ These thresholds are context-dependent. A d = 0.2 in a medical trial might be huge. In product analytics, d = 0.5 might not justify a full sprint.


In [ ]:
def effect_sizes(a: np.ndarray, b: np.ndarray):
    """Returns Cohen's d and Hedges' g for two independent groups."""
    n1, n2 = len(a), len(b)
    var1, var2 = np.var(a, ddof=1), np.var(b, ddof=1)

    # Pooled standard deviation
    s_pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    d = (np.mean(a) - np.mean(b)) / s_pooled

    # Hedges' g correction factor (small-sample bias correction)
    J = 1 - (3 / (4 * (n1 + n2) - 9))
    g = d * J

    return d, g

d, g = effect_sizes(app_ratings, play_ratings)

# Interpret
def interpret_d(val):
    v = abs(val)
    if v < 0.2:  return "negligible"
    if v < 0.5:  return "small"
    if v < 0.8:  return "medium"
    return "large"

print("Effect Size Results")
print("─" * 35)
print(f"  Cohen's d  = {d:.4f}  ({interpret_d(d)} effect)")
print(f"  Hedges' g  = {g:.4f}  ({interpret_d(g)} effect)")
print()
print("Putting it together:")
print(f"  Statistically significant?  {'Yes' if p_value < ALPHA else 'No'}  (p = {p_value:.4f})")
print(f"  Practically meaningful?     {interpret_d(d).title()} effect (|d| = {abs(d):.3f})")


---
## Step 8 — The Confidence Interval: *How Big Is the Gap, Really?*

The p-value gives you a binary signal: significant or not. The **confidence interval** (CI) gives you something richer — a *range* for where the true difference probably lives.

> A 95% CI means: if we ran this study 100 times, roughly 95 of those intervals would contain the true population difference.

**The key diagnostic:** if the 95% CI includes zero, then zero is a plausible value for the true difference — which is consistent with failing to reject H₀. If the interval is entirely on one side of zero, we have directional evidence.

The CI also anchors the result in practical terms. Instead of just "there's a significant difference", you can say "the true difference is most likely somewhere between X and Y stars" — which is far more useful for product decisions.


In [ ]:
# ── 95% CI using Welch-Satterthwaite ─────────────────────────────────────
n1, n2       = len(app_ratings), len(play_ratings)
mean_diff    = np.mean(app_ratings) - np.mean(play_ratings)
var1, var2   = np.var(app_ratings, ddof=1), np.var(play_ratings, ddof=1)

se           = np.sqrt(var1/n1 + var2/n2)

# Welch-Satterthwaite degrees of freedom
df_w = (var1/n1 + var2/n2)**2 / (
    (var1/n1)**2 / (n1 - 1) + (var2/n2)**2 / (n2 - 1)
)

t_crit = stats.t.ppf(0.975, df=df_w)   # two-tailed 95%
ci_low  = mean_diff - t_crit * se
ci_high = mean_diff + t_crit * se

print(f"Mean difference (App Store − Play Store) : {mean_diff:+.4f} stars")
print(f"Standard error of the difference         : {se:.4f}")
print(f"Welch degrees of freedom                 : {df_w:.1f}")
print(f"95% Confidence Interval                  : [{ci_low:.4f},  {ci_high:.4f}]")
print()
if ci_high < 0:
    print("✅ Entire CI is negative → App Store ratings are consistently lower.")
elif ci_low > 0:
    print("✅ Entire CI is positive → App Store ratings are consistently higher.")
else:
    print("🚫 CI includes zero → a true difference of zero is still plausible.")


In [ ]:
# ── Figure 3: CI visualisation ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.5))

# Zero line
ax.axvline(0, color="tomato", linestyle="--", linewidth=2,
           label="Zero (H₀: no difference)", zorder=3)

# CI error bar
ax.errorbar(
    mean_diff, 0,
    xerr=[[mean_diff - ci_low], [ci_high - mean_diff]],
    fmt="o", color="#2c3e50", ecolor="#2c3e50",
    elinewidth=4, capsize=14, capthick=2.5, markersize=12,
    label=f"Mean diff = {mean_diff:.3f} stars\n95% CI  [{ci_low:.3f}, {ci_high:.3f}]",
    zorder=4
)

# Shade the CI region
ax.axvspan(ci_low, ci_high, alpha=0.12, color="#2c3e50", label="95% CI region")

ax.set_xlabel("Difference in Mean Rating  (App Store − Play Store)", fontsize=12)
ax.set_yticks([])
ax.set_title("95% Confidence Interval for the Difference in Means", fontsize=13, pad=12)
ax.legend(fontsize=10, loc="upper left")
ax.set_xlim(min(ci_low - 0.15, -0.2), max(ci_high + 0.15, 0.2))
plt.tight_layout()
plt.show()

print()
print("Reading the chart:")
print("  • The dot is our observed mean difference")
print("  • The whiskers span the 95% confidence interval")
print("  • If the red dashed line (zero) is inside the whiskers → H₀ is still plausible")
print("  • If the red line is outside → consistent with rejecting H₀")


---
## Step 9 — The Permutation Test: Engineering a P-Value from Scratch 🔧

This is the most important section in the notebook. Everything so far has relied on a formula. Now we're going to *build* a p-value by simulation — and in doing so, you'll understand what it actually means at the deepest level.

---

### The core idea

The t-test gives us a p-value by comparing our t-statistic to a mathematical distribution. But where does that distribution come from? What does "the null hypothesis is true" actually look like?

Here's the insight: **if there's truly no difference between platforms, then the label "App Store" or "Play Store" on any given review is meaningless.** We could shuffle the labels randomly and the resulting difference would be equally valid.

So we do exactly that. 10,000 times.

1. Compute the observed mean difference (our actual result)
2. Pool all ratings together and shuffle the platform labels
3. Recompute the mean difference in this shuffled world
4. Repeat 10,000 times to build the *null distribution*
5. Count how many shuffled differences are as extreme as our observed one
6. That proportion **is the p-value** — no formula needed

This is called a **permutation test** (also a randomisation test). It's more flexible than the t-test, makes fewer assumptions, and directly answers the question: *"how often would random chance produce a result this extreme?"*

> For an ML engineer, building simulation-based tests is a core skill. You'll use this pattern again when you bootstrap confidence intervals, validate models with permutation feature importance, and run A/B test analysis.


In [ ]:
def permutation_test(
    a: np.ndarray,
    b: np.ndarray,
    n_permutations: int = 10_000,
    seed: int = 42,
) -> tuple[float, np.ndarray, float]:
    """
    Runs a two-sided permutation test comparing the means of two groups.
    
    Returns: (observed_diff, null_distribution, permutation_p_value)
    """
    rng          = np.random.default_rng(seed)
    observed_diff = np.mean(a) - np.mean(b)
    combined     = np.concatenate([a, b])
    n_a          = len(a)
    null_diffs   = np.empty(n_permutations)

    for i in range(n_permutations):
        rng.shuffle(combined)
        null_diffs[i] = np.mean(combined[:n_a]) - np.mean(combined[n_a:])

    # Two-tailed: how many shuffled diffs are *at least as extreme* as observed?
    p_perm = (np.abs(null_diffs) >= np.abs(observed_diff)).mean()

    return observed_diff, null_diffs, p_perm


obs_diff, null_dist, p_perm = permutation_test(app_ratings, play_ratings, n_permutations=N_PERMUTATIONS)

print(f"Observed mean difference : {obs_diff:.4f} stars")
print(f"Permutation p-value      : {p_perm:.4f}")
print(f"Welch t-test p-value     : {p_value:.4f}  ← should be similar!")
print()
print("The two methods agree (roughly) because both are answering the same underlying question.")
print("The permutation test just answers it by simulation rather than by formula.")


In [ ]:
# ── Figure 4: Null distribution with observed result marked ──────────────
fig, ax = plt.subplots(figsize=(11, 5))

# Histogram of null distribution
ax.hist(null_dist, bins=60, color="#cbd5e0", edgecolor="white",
        linewidth=0.5, zorder=2, label="Null distribution\n(shuffled labels)")

# Shade the rejection region (values at least as extreme as observed)
extreme_mask = np.abs(null_dist) >= np.abs(obs_diff)
ax.hist(null_dist[extreme_mask], bins=60,
        color="tomato", alpha=0.7, edgecolor="white", linewidth=0.5,
        zorder=3, label=f"As extreme as observed\n({extreme_mask.sum()} / 10,000 = {p_perm:.3f})")

# Mark the observed difference (both sides, two-tailed)
ax.axvline( obs_diff, color="#e07b54", linestyle="--", linewidth=2.5,
            label=f"Observed diff = {obs_diff:.3f}")
ax.axvline(-obs_diff, color="#e07b54", linestyle="--", linewidth=2.5, alpha=0.5)

# Annotate
ax.annotate(
    f"Observed\ndifference\n({obs_diff:.3f})",
    xy=(obs_diff, 0), xytext=(obs_diff + 0.05, ax.get_ylim()[1] * 0.5),
    fontsize=9, color="#e07b54",
    arrowprops=dict(arrowstyle="->", color="#e07b54", lw=1.5)
)

ax.set_title("Permutation Test: What Does Random Chance Look Like?", fontsize=13, pad=12)
ax.set_xlabel("Mean Difference (App Store − Play Store) under Shuffled Labels", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.legend(fontsize=10, loc="upper left")
plt.tight_layout()
plt.show()

print()
print("How to read this chart:")
print("  • The grey histogram = the 'null world': differences you'd see if labels didn't matter")
print("  • The red bars = results as extreme as ours (proportion = permutation p-value)")
print("  • The dashed lines = our actual observed difference")
print("  • If the dashed line is deep in the tail → our result is unusual in the null world → reject H₀")


---
## Step 10 — Statistical Power: Why Sample Size Matters

Here's a scenario that trips people up constantly:

> *"We ran the test on 30 reviews and got p = 0.12, so there's no difference."*

That conclusion may be completely wrong. With only 30 samples, you might lack the **statistical power** to detect a real difference even if it exists. A non-significant result with small n tells you almost nothing.

**Statistical power** is the probability of correctly detecting a real effect when it exists (avoiding a Type II error — a false negative). It depends on:
- Effect size (bigger effects are easier to detect)
- Sample size (more data = more power)
- Significance level α

The experiment below demonstrates this directly: we subsample different numbers of reviews from our dataset and run the t-test on each. Watch how detection rate changes with sample size.


In [ ]:
def power_by_sample_size(
    a: np.ndarray,
    b: np.ndarray,
    sample_sizes: list,
    n_trials: int = 500,
    alpha: float = 0.05,
    seed: int = 0,
) -> dict:
    """
    For each sample size n, draw n samples from each group n_trials times and
    record what proportion of t-tests return p < alpha.
    That proportion estimates the statistical power at that sample size.
    """
    rng     = np.random.default_rng(seed)
    results = {}

    for n in sample_sizes:
        p_vals = []
        for _ in range(n_trials):
            sample_a = rng.choice(a, size=min(n, len(a)), replace=False)
            sample_b = rng.choice(b, size=min(n, len(b)), replace=False)
            _, pv    = stats.ttest_ind(sample_a, sample_b, equal_var=False)
            p_vals.append(pv)
        results[n] = np.mean(np.array(p_vals) < alpha)

    return results

sample_sizes = [15, 25, 40, 60, 80, 100, 130, 160, 200, 250]
power_results = power_by_sample_size(app_ratings, play_ratings, sample_sizes, n_trials=N_POWER_TRIALS)

print(f"{'Sample size (per platform)':>28} | {'Detection rate':>16} | {'Verdict':>20}")
print("─" * 70)
for n, rate in power_results.items():
    verdict = "Rarely detects" if rate < 0.3 else ("Sometimes" if rate < 0.6 else ("Often" if rate < 0.8 else "Reliably detects"))
    print(f"{n:>28} | {rate:>14.1%} | {verdict:>20}")


In [ ]:
# ── Figure 5: Power curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ns    = list(power_results.keys())
power = list(power_results.values())

ax.plot(ns, power, "o-", color="#5b8db8", linewidth=2.5, markersize=8, zorder=3)
ax.fill_between(ns, power, alpha=0.15, color="#5b8db8")

# 80% power guideline (conventional minimum for well-designed studies)
ax.axhline(0.80, color="tomato", linestyle="--", linewidth=1.8,
           label="80% power (conventional target)")
ax.axhline(ALPHA, color="grey",  linestyle=":",  linewidth=1.5,
           label=f"α = {ALPHA} (false positive rate)")

# Shade the "underpowered" zone
ax.axhspan(0, 0.80, alpha=0.06, color="tomato")
ax.text(18, 0.40, "Underpowered zone\n(high false negative risk)",
        color="tomato", fontsize=9.5, alpha=0.8)

ax.set_xlabel("Sample Size per Platform", fontsize=12)
ax.set_ylabel("Proportion of Tests with p < 0.05", fontsize=12)
ax.set_title("Statistical Power Curve — How Many Reviews Do We Need?", fontsize=13, pad=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

print()
print("Key takeaway:")
print("  Low sample sizes don't prove there's no difference.")
print("  They just mean you can't reliably detect it even if it exists.")
print("  'Not significant' ≠ 'no effect'.")


### 🔧 Have a go: Watch the p-value wobble with small samples

Run the cell below. It draws a fresh random subsample of  reviews from each platform and runs the t-test live. Change  and rerun it several times — notice how the p-value jumps around at small n, and stabilises at large n. This is statistical power in action.


In [ ]:
# ── 🔧 Interactive subsampling demo ──────────────────────────────────────
# Change n and rerun this cell multiple times to see p-value variability

n = 40   # ← try: 15, 30, 40, 80, 150, 250

rng_demo = np.random.default_rng()   # no seed = different each run

sample_app  = rng_demo.choice(app_ratings,  size=min(n, len(app_ratings)),  replace=False)
sample_play = rng_demo.choice(play_ratings, size=min(n, len(play_ratings)), replace=False)

t_demo, p_demo = stats.ttest_ind(sample_app, sample_play, equal_var=False)

print(f"Sample size    : n = {n} per platform")
print(f"App Store mean : {sample_app.mean():.3f}   Play Store mean : {sample_play.mean():.3f}")
print(f"Observed diff  : {sample_app.mean() - sample_play.mean():+.3f} stars")
print(f"t-statistic    : {t_demo:.3f}")
print(f"p-value        : {p_demo:.4f}  {'✅ significant' if p_demo < ALPHA else '🚫 not significant'} at α = {ALPHA}")
print()
print("↑ Rerun this cell a few times. Notice how the result changes at small n.")
print("  At large n it stabilises — that's statistical power working.")

# Visualise this one sample
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, sample, platform in zip(axes, [sample_app, sample_play], ["App Store", "Play Store"]):
    counts = pd.Series(sample).value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=COLOURS[platform], alpha=0.85, edgecolor="white")
    ax.axvline(sample.mean(), color="black", linestyle="--", linewidth=2,
               label=f"Mean = {sample.mean():.2f}")
    ax.set_title(f"{platform}  (n={n})", fontweight="bold", color=COLOURS[platform])
    ax.set_xticks([1,2,3,4,5])
    ax.set_xlabel("Star Rating")
    ax.legend()
plt.suptitle(f"One random subsample of n={n} — your result will vary!", fontsize=12)
plt.tight_layout()
plt.show()


---
## Step 11 — The Final Verdict

Time to pull everything together and answer Priya's question in a way that's honest, precise, and actually actionable.

A good statistical summary for a stakeholder should include:
- The observed difference (the "what")
- Whether it's significant and what the p-value was (the "is it real?")
- The effect size (the "does it matter?")
- The confidence interval (the "how big could it actually be?")
- A plain-English recommendation (the "so what?")

Stats answers one question. Product work answers the rest.


In [ ]:
# ── Traffic-light summary ────────────────────────────────────────────────
is_sig = p_value < ALPHA
effect_label = interpret_d(g)

print("━" * 62)
print("   📋  TEATIME TYCOON — RATINGS ANALYSIS REPORT")
print("━" * 62)
print(f"   App Store   : mean = {np.mean(app_ratings):.3f} ± {np.std(app_ratings, ddof=1):.3f}  (n = {n1})")
print(f"   Play Store  : mean = {np.mean(play_ratings):.3f} ± {np.std(play_ratings, ddof=1):.3f}  (n = {n2})")
print(f"   Difference  : {mean_diff:+.3f} stars  (App Store − Play Store)")
print()
print(f"   Method      : Welch's independent samples t-test (two-tailed)")
print(f"   t-statistic : {t_stat:.3f}")
print(f"   p-value     : {p_value:.4f}  (permutation test: {p_perm:.4f})")
print(f"   95% CI      : [{ci_low:.3f},  {ci_high:.3f}]")
print(f"   Hedges' g   : {g:.3f}  →  {effect_label} effect")
print()
print("─" * 62)

if is_sig and abs(g) >= 0.5:
    print("   🔴  RESULT: Significant AND practically meaningful")
    print()
    print("   App Store users are rating TeaTime Tycoon lower, and the gap")
    print("   is large enough to be worth acting on. Priya is right.")
    print("   Recommendation: Escalate to the iOS team. Dig into 1–2 star")
    print("   review text for specific pain points before planning the fix.")

elif is_sig:
    print("   🟡  RESULT: Statistically significant, but modest effect size")
    print()
    print("   The gap is real — it's unlikely to be random noise. But with")
    print(f"   Hedges' g = {g:.2f}, the difference is small in practical terms.")
    print("   Recommendation: Don't panic-rewrite the iOS app. Monitor the")
    print("   trend over the next sprint and check for version-specific issues.")

else:
    print("   🟢  RESULT: Not statistically significant")
    print()
    print("   We can't confidently claim a real difference exists. The gap")
    print("   Priya spotted is plausibly just random sampling variation.")
    print("   Recommendation: Tell Priya to hold off on the iOS redesign.")
    print("   Collect more reviews and rerun the analysis in 2–3 weeks.")

print("━" * 62)


---
## Step 12 — Beyond the Numbers: Confounders and Caveats

A statistically significant result doesn't mean you've found the *cause*. Even a well-run test can mislead you if there are confounding variables — factors that differ between the groups and independently affect ratings.

For our TeaTime Tycoon analysis, worth considering:

**Review prompting:** App Store and Play Store use different prompts to ask users for reviews. App Store prompts after in-app events; Play Store has slightly different timing. This could bias ratings independently of app quality.

**App version:** Are both platforms on the same version? If the iOS build has a recent regression, that explains the gap without any platform-intrinsic bias.

**Demographic mix:** Different platforms attract different users in different regions. Android dominates globally; iPhone skews toward certain demographics. A rating difference could reflect user mix, not app quality.

**Review self-selection:** Users who bother to leave a review are not a random sample. If one platform's power users are more likely to leave negative reviews, that contaminates the comparison.

**Time window:** Are we comparing the same date range? A viral complaint thread on Reddit could spike 1-star reviews on one platform for a week.

> The rule: **statistics can tell you *that* something differs. Only domain knowledge and further investigation can tell you *why*.**


---
## ✅ Wrap-Up: What You've Done

You've run a complete, professional-grade hypothesis test from raw data to business recommendation. Here's the full pipeline you built:

| Step | What you did |
|---|---|
| 1 | Built a portable data pipeline (real or synthetic CSVs) |
| 2 | Validated data quality before touching any analysis |
| 3 | Visualised distributions before running any test |
| 4 | Set formal hypotheses and α *before* seeing results |
| 5 | Ran Welch's t-test with the right default assumptions |
| 6 | Correctly interpreted the p-value (and avoided the myths) |
| 7 | Measured effect size with Cohen's d and Hedges' g |
| 8 | Built and visualised a confidence interval |
| 9 | Built a permutation test from scratch to earn the intuition |
| 10 | Demonstrated statistical power through subsampling |
| 11 | Translated everything into a plain-English recommendation |
| 12 | Identified confounders and caveats a real analyst would flag |

---

## 🔧 Try It Yourself

These exercises are designed to feel like real analytics work, not a maths exam.

**1. One-sided hypothesis**  
Priya is specifically worried that App Store ratings are *lower*. Change the test to one-sided (`alternative='less'` in `ttest_ind`) and rerun. Does the p-value change? Why?

**2. Vary α**  
Change `ALPHA` to 0.01 and rerun the verdict cell. Does your conclusion change? What does this tell you about the arbitrariness of significance thresholds?

**3. What if the gap vanished?**  
Change `app_probs` in the data generator so both platforms have identical distributions. Rerun everything. What happens to p, d, and the CI?

**4. Mann-Whitney U (non-parametric alternative)**  
Star ratings are technically ordinal — the "distance" between 3 and 4 stars isn't necessarily the same as between 4 and 5. The t-test treats them as continuous. Try `scipy.stats.mannwhitneyu(app_ratings, play_ratings, alternative='two-sided')`. Does it agree with the t-test?

**5. Bootstrap the confidence interval**  
Instead of the Welch-Satterthwaite formula in Step 8, estimate the 95% CI by resampling with replacement 10,000 times. Compare the result to the formula-based CI. Do they agree?

**6. Segment by time**  
The dataset has dates. Split each platform's reviews into "first 45 days" vs "last 45 days" and run the t-test within each window. Has the gap been stable over time, or is it driven by a recent event?

**7. Three platforms (ANOVA teaser)**  
Imagine TeaTime Tycoon also launches on the Amazon Appstore. Why can't you just run three t-tests (App vs Play, App vs Amazon, Play vs Amazon)? Look up the **multiple comparisons problem** and **Bonferroni correction**. What would you use instead?
